In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1) 데이터 로드
df = pd.read_csv("train.csv")

# 2) 간단한 EDA
print("\n결측치:\n", df.isnull().sum().sort_values(ascending=False).head(10))
print("\n타겟 비율:\n", df["Survived"].value_counts(normalize=True))

# 3) Name 컬럼 추가
use_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Name"]
df = df[use_cols].copy()

y = df["Survived"]
X = df.drop(columns=["Survived"])

# 4) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# (A) 컬럼 구분
num_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Sex", "Embarked"]
text_col = "Name"

# (B) 수치형 파이프라인
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# (C) 범주형 파이프라인
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# (D) 텍스트 파이프라인
text_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(min_df=2))
])

# (E) 전처리 통합
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
    ("txt", text_pipeline, text_col)
])

# (F) 전체 Pipeline 구성 (로지스틱 회귀로 변경)
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=3000, random_state=42))
])

# 8) GridSearchCV (C만 튜닝)
param_grid = {
    "clf__C": [0.01, 0.1, 1, 10]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBest Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)


결측치:
 Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
dtype: int64

타겟 비율:
 Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Best Params: {'clf__C': 1}
Best CV Score: 0.8161036146951639


In [2]:
# 9) test 평가
best_model = grid.best_estimator_
pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, pred))
print("\nClassification Report:\n", classification_report(y_test, pred))


Test Accuracy: 0.8324022346368715

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.87       110
           1       0.82      0.72      0.77        69

    accuracy                           0.83       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179



In [3]:
# 여기서도 확률값 뽑아보기
best_model.predict_proba(X_test)[0]

array([0.96687557, 0.03312443])

In [4]:
best_model.predict(X_test)[0]

np.int64(0)